In [ ]:
import pandas as pd
import requests
import time
from bs4 import BeautifulSoup
import lyricsgenius

# Get a free token at https://genius.com/api-clients -> "New API Client"
# (any app name/URL is fine) -> copy the "Client Access Token" below.
GENIUS_API_TOKEN = "PASTE_YOUR_GENIUS_TOKEN_HERE"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
}


def get_billboard_year_end_list(year):
    """
    Scrapes the Billboard Year-End Hot 100 songs and their artists from the specified year.
    """
    url = "https://www.billboard.com/charts/year-end/" + year + "/hot-100-songs"
    print(url)
    page = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(page.content, 'html.parser')

    # Extract song titles — only keep ones that live inside an actual chart
    # row (<li class="o-chart-results-list__item">). This excludes sidebar/
    # widget text (e.g. "Gains in Weekly Performance") that also uses the
    # c-title class but isn't part of the chart itself.
    results_songs = soup.find_all('h3', attrs={'class': 'c-title'})
    songs = [
        result.getText().strip()
        for result in results_songs
        if result.find_parent('li', class_='o-chart-results-list__item')
    ]
    songs = songs[0:100]
    print(songs)

    # Extract artist names — same row-scoping applied here
    results_artists = soup.find_all('span', attrs={'class': 'c-label'})
    artists = [
        result.getText().strip()
        for result in results_artists
        if result.find_parent('li', class_='o-chart-results-list__item')
        and not result.getText().strip().isdigit()
    ]
    artists = artists[0:100]

    # Handle collaboration artist names
    fixed_artists = []
    for weird_format in artists:
        if " X " in weird_format or " & " in weird_format:
            fix_format = weird_format.replace(" X ", " & ")
            fixed_artists.append(fix_format)
        else:
            fixed_artists.append(weird_format)

    print(len(songs), len(fixed_artists))
    if len(songs) != len(fixed_artists):
        print("WARNING: song/artist counts don't match — results may be misaligned. "
              "Inspect the c-label selector before trusting this data.")

    # Create DataFrame
    hot100_data_frame = pd.DataFrame({'artist': fixed_artists, 'song': songs})

    return hot100_data_frame


def search_lyrics(genius, song_name, artist_name):
    """
    Searches for lyrics via Genius. We call the official, token-authenticated
    Genius search API (api.genius.com/search) directly with `requests`,
    rather than using lyricsgenius's own search_song(), because that method
    routes through an undocumented public endpoint (genius.com/api/search)
    that's known to return 403s for non-browser traffic regardless of token.
    Once we have the song's Genius page URL, lyricsgenius's lyrics scraper
    (genius.lyrics()) handles pulling the actual lyrics text, which works
    fine — it's only the search step that's broken.
    """
    primary_artist = artist_name.split(" & ")[0].split(" X ")[0]
    print(f"Searching for lyrics for: {song_name} by {artist_name}")

    try:
        search_url = "https://api.genius.com/search"
        params = {"q": f"{song_name} {primary_artist}"}
        auth_headers = {"Authorization": f"Bearer {GENIUS_API_TOKEN}"}
        response = requests.get(search_url, params=params, headers=auth_headers)
        response.raise_for_status()
        hits = response.json()["response"]["hits"]

        if not hits:
            print(f"No lyrics found for {song_name} by {artist_name}")
            return None

        song_url = hits[0]["result"]["url"]
        lyrics_page = requests.get(song_url, headers=HEADERS)
        lyrics_page.raise_for_status()
        lyrics_soup = BeautifulSoup(lyrics_page.content, 'html.parser')

        containers = lyrics_soup.find_all('div', attrs={'data-lyrics-container': 'true'})
        if not containers:
            print(f"No lyrics found for {song_name} by {artist_name}")
            return None

        lines = []
        for container in containers:
            for br in container.find_all('br'):
                br.replace_with('\n')
            lines.append(container.get_text())
        return '\n'.join(lines).strip()
    except Exception as e:
        print(f"Error fetching lyrics for {song_name} by {artist_name}: {e}")
        return None


if __name__ == "__main__":
    # Request input for the year
    year = input('Enter the year: ')

    # Get the Billboard Year-End List for the specified year
    year_end_list = get_billboard_year_end_list(year)

    # Set up the Genius client once, outside the loop
    genius = lyricsgenius.Genius(GENIUS_API_TOKEN)
    genius.verbose = False                 # don't print lyricsgenius's own status messages
    genius.remove_section_headers = True   # strip [Chorus], [Verse 1], etc.
    genius.skip_non_songs = True           # skip things like track lists, tabs, etc.
    genius.excluded_terms = ["(Remix)", "(Live)"]

    # If the DataFrame is not empty, process the lyrics
    if not year_end_list.empty:
        # "w" so reruns overwrite instead of appending duplicates
        with open(f"{year}.lyrics.txt", "w", encoding='utf-8') as lyrics_file:
            for index, row in year_end_list.iterrows():
                lyrics = search_lyrics(genius, row["song"], row["artist"])
                if lyrics:
                    lyrics_file.write(f"\n\n{row['artist']} - {row['song']}\n\n{lyrics}\n")
                else:
                    print(f"Lyrics not found for {row['artist']} - {row['song']}")

                # Small delay to stay well within Genius's rate limits
                time.sleep(2)
        print(f'Lyrics for {year} year-end list saved!')
    else:
        print('No data available for the specified year.')

In [ ]:
song = genius.search_song("Flowers", "Miley Cyrus")


In [ ]:
import re
import os
import matplotlib.pyplot as plt
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from wordcloud import WordCloud

#years = ["2019", "2020", "2021", "2022"]
#for year in years:
 #   f = open(year + '.lyrics.txt', 'rb')
  #  all_words = ''
   # for sentence in f.readlines():
    #    this_sentence = sentence.decode('utf-8')
     #   all_words += this_sentence
   # f.close()

    #remove identifiers like chorus, verse, etc
    #all_words = re.sub(r'[\(\[].*?[\)\]]', '', all_words)
    #remove empty lines
    #all_words = os.linesep.join([s for s in all_words.splitlines() if s])
    
    #f = open(year + '.lyrics.cleaned.txt', 'wb')
    #f.write(all_words.encode('utf-8'))
   # f.close()

import re
import os

years = ["2023", "2024", "2025"]

for year in years:
    f = open(year + '.lyrics.txt', 'rb')
    all_words = ''
    for sentence in f.readlines():
        this_sentence = sentence.decode('utf-8')
        
        # Exclude lines with artist and song information
        if not re.match(r'\d+\.\s+[A-Za-z0-9\s&\(\)\[\]\-.,]+', this_sentence):
            all_words += this_sentence

    f.close()

    # Remove identifiers like chorus, verse, etc
    all_words = re.sub(r'[\(\[].*?[\)\]]', '', all_words)
    # Remove empty lines
    all_words = os.linesep.join([s for s in all_words.splitlines() if s])
    
    f = open(year + '.lyrics.cleaned.txt', 'wb')
    f.write(all_words.encode('utf-8'))
    f.close()
    words = all_words.split(" ")
    filtered_words = [word for word in words if word not in stopwords.words('english') and len(word) > 1 and word not in ['na','la']] # remove the stopwords
    joined_words = " ".join(filtered_words)
        # Generate a word cloud image
    wordcloud = WordCloud().generate(joined_words)

    # Display the generated image:
    # the matplotlib way:
    import matplotlib.pyplot as plt
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")

    # lower max_font_size
    wordcloud = WordCloud(max_font_size=40).generate(joined_words)
    plt.figure()
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.show()

    # The pil way (if you don't have matplotlib)
    # image = wordcloud.to_image()
    # image.show()
    plt.savefig(year + '_wordcloud_lyrics.png')

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk
nltk.download('vader_lexicon')
import pandas as pd
df = pd.DataFrame(columns=('year', 'pos', 'neu', 'neg'))
sid = SentimentIntensityAnalyzer()
i=0
years = ["2023", "2024", "2025"]
for year in years:
    num_positive = 0
    num_negative = 0
    num_neutral = 0

    f = open(year + ".lyrics.cleaned.txt", "rb")
    for sentence in f.readlines():
        this_sentence = sentence.decode('utf-8')
        comp = sid.polarity_scores(this_sentence)
        comp = comp['compound']
        if comp >= 0.5:
            num_positive += 1
        elif comp > -0.5 and comp < 0.5:
            num_neutral += 1
        else:
            num_negative += 1

    num_total = num_negative + num_neutral + num_positive
    percent_negative = (num_negative/float(num_total))*100
    percent_neutral = (num_neutral/float(num_total))*100
    percent_positive = (num_positive/float(num_total))*100
    df.loc[i] = (year, percent_positive, percent_neutral, percent_negative)
    i+=1
    #print(year, ": ", "positive: ", num_positive, "neutral: ", num_neutral, "negative: ", num_negative)
    print(year, ": ", "positive: ", percent_positive, "% ",  num_positive, "neutral: ", percent_neutral, "% ", num_neutral,  "negative: ", percent_negative, "% ", num_negative)
                 
df.plot.bar(x='year', stacked=True)
plt.show()     